# CLICK Trash Detection: YOLO training at 640

Trains Ultralytics YOLO object detectors on the CLICK Trash Detection Dataset for every combination of

- **label space**: `single class` (trash / not trash) and `multi class` (13 material classes)
- **model size**: `n`, `s`, `m`, `l`, `x`

at `imgsz=640`, then evaluates each best checkpoint on the held-out **test** split and collects the results in one table.

The notebook lives in `Notebooks/` and expects the dataset at `Dataset/01 Full Dataset/` in the repository root, one level up. Edit `DATASET_ROOT` in the configuration cell if your copy is elsewhere.
The same logic is available as a command-line script in `Notebooks/train_yolo.py`.

In [ ]:
# Install once if needed (uncomment)
# %pip install -q ultralytics pandas matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
import time
import pandas as pd
import torch
from ultralytics import YOLO

# this notebook lives in <repo>/Notebooks; the dataset in <repo>/Dataset/01 Full Dataset
HERE = Path.cwd()
REPO_ROOT = HERE.parent if HERE.name == "Notebooks" else HERE
DATASET_ROOT = REPO_ROOT / "Dataset" / "01 Full Dataset"
SPACES = {
    "single": DATASET_ROOT / "single class" / "data.yaml",
    "multi":  DATASET_ROOT / "multi class"  / "data.yaml",
}

# ---- what to train -------------------------------------------------------
LABEL_SPACES = ["single", "multi"]       # subset of ["single", "multi"]
SIZES        = ["n", "s", "m", "l", "x"] # YOLO calls the largest size "x" (often spoken of as "xl")
FAMILY       = "yolo11"                  # "yolo11", "yolov8", "yolo12", ...
IMGSZ        = 640
EPOCHS       = 100
BATCH        = -1                        # -1 = let Ultralytics choose a batch that fits the GPU
PATIENCE     = 30                        # early stopping
WORKERS      = 4
SEED         = 42
FRACTION     = 1.0                       # <1.0 trains on a subset, handy for a quick smoke test
PROJECT      = REPO_ROOT / "runs" / "detect"  # every run goes under here
WORKSPACE    = REPO_ROOT / "runs" / "workspace"  # linked images/ + labels/ per label space
EVAL_ON_TEST = True
SKIP_EXISTING = True                     # skip runs whose best.pt already exists

for s in LABEL_SPACES:
    assert SPACES[s].exists(), f"missing {SPACES[s]}"
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("WARNING: no CUDA GPU, training on CPU will be very slow")
print("runs:", [(s, f"{FAMILY}{z}") for s in LABEL_SPACES for z in SIZES])

## 2. Training and test evaluation

One run per (label space, size). The dataset keeps one shared `images/` folder and one `labels/` folder per label space; Ultralytics needs the two side by side, so each run first builds `runs/workspace/<space>/` with one hard link or symlink per file (no photo is copied) and trains from a generated `data.yaml` there. Each run writes weights, curves and confusion matrices to `runs/detect/<space>_<family><size>/`.
The test evaluation of each run goes to `runs/detect/<space>_<family><size>_test/`.
The results table is saved after every run, so an interrupted session still leaves a usable `summary.csv`.

In [ ]:
import os, shutil, sys, yaml

_LINK_MODE = {}

def link_file(src: Path, dst: Path) -> None:
    """Make dst refer to file src without copying its bytes: hard link, else symlink, else copy."""
    if dst.exists():
        return
    for mode in ("hardlink", "symlink", "copy"):
        if _LINK_MODE and _LINK_MODE["mode"] != mode:
            continue
        try:
            if mode == "hardlink":
                os.link(src, dst)
            elif mode == "symlink":
                dst.symlink_to(src)
            else:
                shutil.copy2(src, dst)
            if not _LINK_MODE:
                _LINK_MODE["mode"] = mode
                print("workspace files are", "copies" if mode == "copy" else mode + "s", "of the dataset files")
            return
        except OSError:
            continue
    raise RuntimeError(f"could not link or copy {src} -> {dst}")


def prepare_workspace(space: str) -> Path:
    """The dataset has one shared images/ folder and one labels/ folder per label space.
    Ultralytics derives label paths from the *resolved* image folder, so a linked folder is
    followed to the real photos and the labels are then searched next to them. Real folders
    holding one link per file do work, so build <repo>/runs/workspace/<space>/{images,labels}
    that way (no photo is copied with hard links or symlinks) and write a data.yaml there."""
    src_yaml = SPACES[space]
    ws = WORKSPACE / space
    for split in ("train", "val", "test"):
        for kind, src_dir, ext in (("images", DATASET_ROOT / "images" / split, ".jpg"),
                                   ("labels", src_yaml.parent / "labels" / split, ".txt")):
            dst_dir = ws / kind / split
            dst_dir.mkdir(parents=True, exist_ok=True)
            for f in src_dir.iterdir():
                if f.suffix.lower() == ext:
                    link_file(f, dst_dir / f.name)
    cfg = yaml.safe_load(src_yaml.read_text(encoding="utf-8"))
    cfg.update({"train": "images/train", "val": "images/val", "test": "images/test"})
    cfg.pop("path", None)
    out = ws / "data.yaml"
    out.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    for split in ("train", "val", "test"):
        n_img = len(list((ws / "images" / split).glob("*.jpg")))
        n_lbl = len(list((ws / "labels" / split).glob("*.txt")))
        assert n_img and n_img == n_lbl, f"{space}/{split}: {n_img} images, {n_lbl} labels"
    return out


def train_and_eval(space: str, size: str) -> dict:
    data_yaml = prepare_workspace(space)
    run_name = f"{space}_{FAMILY}{size}"
    best = PROJECT / run_name / "weights" / "best.pt"
    print(f"
{'='*78}
{run_name}
{'='*78}")
    t0 = time.time()

    if SKIP_EXISTING and best.exists():
        print("best.pt exists, skipping training:", best)
    else:
        YOLO(f"{FAMILY}{size}.pt").train(
            data=str(data_yaml), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
            patience=PATIENCE, workers=WORKERS, seed=SEED, fraction=FRACTION,
            project=str(PROJECT), name=run_name, exist_ok=True, plots=True,
        )

    row = {"space": space, "model": f"{FAMILY}{size}", "imgsz": IMGSZ,
           "train_minutes": round((time.time() - t0) / 60, 1)}

    if EVAL_ON_TEST and best.exists():
        trained = YOLO(str(best))
        row["params_M"] = round(sum(p.numel() for p in trained.model.parameters()) / 1e6, 2)
        m = trained.val(data=str(data_yaml), split="test", imgsz=IMGSZ,
                        batch=BATCH if BATCH > 0 else 16,
                        project=str(PROJECT), name=f"{run_name}_test", exist_ok=True, plots=True)
        row.update({
            "test_mAP50-95": round(float(m.box.map), 4),
            "test_mAP50":    round(float(m.box.map50), 4),
            "test_precision": round(float(m.box.mp), 4),
            "test_recall":   round(float(m.box.mr), 4),
            "inference_ms":  round(float(m.speed.get("inference", 0)), 2),
        })
    row["best_pt"] = str(best)
    return row


rows = []
for space in LABEL_SPACES:
    for size in SIZES:
        rows.append(train_and_eval(space, size))
        PROJECT.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(PROJECT / "summary.csv", index=False)

results = pd.DataFrame(rows)
results

## 3. Results

In [ ]:
results = pd.read_csv(PROJECT / "summary.csv")
cols = ["space", "model", "params_M", "test_mAP50-95", "test_mAP50", "test_precision", "test_recall", "inference_ms", "train_minutes"]
cols = [c for c in cols if c in results.columns]
display(results[cols].sort_values(["space", "params_M"]).reset_index(drop=True))
(PROJECT / "summary.md").write_text(results[cols].to_markdown(index=False), encoding="utf-8")

In [ ]:
import matplotlib.pyplot as plt

if "test_mAP50-95" in results.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    for space, g in results.groupby("space"):
        g = g.sort_values("params_M")
        ax.plot(g["params_M"], g["test_mAP50-95"], marker="o", label=space)
        for _, r in g.iterrows():
            ax.annotate(r["model"], (r["params_M"], r["test_mAP50-95"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_xlabel("parameters (M)")
    ax.set_ylabel("test mAP50-95")
    ax.set_title(f"{FAMILY} at {IMGSZ}: accuracy vs model size")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Per-run curves

Each run folder holds `results.png` (loss and metric curves), `confusion_matrix.png`, `PR_curve.png` and sample batches. Change `RUN` to inspect a different one.

In [ ]:
from IPython.display import Image, display

RUN = f"{LABEL_SPACES[0]}_{FAMILY}{SIZES[0]}"
for name in ["results.png", "confusion_matrix_normalized.png"]:
    p = PROJECT / RUN / name
    if p.exists():
        display(Image(filename=str(p), width=900))
    else:
        print("not found:", p)

## 5. Quick inference check

Runs the best model of `RUN` on a few test images.

In [ ]:
space = RUN.split("_")[0]
test_dir = DATASET_ROOT / "images" / "test"
imgs = sorted(test_dir.glob("*.jpg"))[:4]
model = YOLO(str(PROJECT / RUN / "weights" / "best.pt"))
for r in model.predict([str(p) for p in imgs], imgsz=IMGSZ, conf=0.25, verbose=False):
    display(r.plot(pil=True).resize((640, 640)))